# 01 — Prepare Training Data

[Notebook guide](README.md) · [Project overview](../README.md) · [English paper](../docs/en/paper.pdf)

Prepare and combine the Serbian instruction/QA sources.

| Item | Details |
| --- | --- |
| Inputs | Two local SQuAD JSON files; Hugging Face Alpaca and Aya datasets. |
| Outputs | `final_train_dataset.json` in the notebook working directory. |
| Requirements | Python with datasets and huggingface_hub; network access and authentication where required. A GPU is not needed. |
| Configuration overview | `base_path` (SQuAD folder); `file_path` (export); sampling seed 42; 500 examples per SQuAD script and 1,000 per Alpaca source. |

**Recorded experiment:** run code cells in their existing order. This presentation edit preserves executable code, parameters, prompts, data, and experimental outputs. Configuration is summarized here without moving assignments. Installation logs are omitted; execution counts belong to the original run. Serbian prompts, reference text, docstrings, and printed outputs remain verbatim.


## 1. Authenticate with Hugging Face

In [1]:
from huggingface_hub import login


HF_TOKEN = ""

login(token=HF_TOKEN)

D:\New folder\faks\MASTER\NLP\projekat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. SQuAD: load and sample

[Source dataset](https://www.kaggle.com/datasets/aleksacvetanovic/squad-sr?resource=download). Inspect one example in each script after sampling.

In [15]:
import json
import os
from datasets import Dataset

base_path = r"C:\Users\Korisnik\.cache\kagglehub\datasets\aleksacvetanovic\squad-sr\versions\1"

# 1. Load SQuAD records
def parse_squad_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        squad_json = json.load(f)
    records = []
    for article in squad_json["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    records.append({
                        "instruction": qa["question"],
                        "input": context,
                        "output": qa["answers"][0]["text"]
                    })
    return Dataset.from_list(records)

# Load and randomly sample SQuAD (500 distinct examples per script)
full_lat = parse_squad_json(os.path.join(base_path, "squad-sr-lat.json")).shuffle(seed=42)
full_cyr = parse_squad_json(os.path.join(base_path, "squad-sr-cyr.json")).shuffle(seed=42)

squad_lat_500 = full_lat.select(range(0, 500))
squad_cyr_500 = full_cyr.select(range(500, 1000))

print(f"SQuAD Latinica: {len(squad_lat_500)} primera")
print(f"SQuAD Ćirilica: {len(squad_cyr_500)} primera")

SQuAD Latinica: 500 primera
SQuAD Ćirilica: 500 primera


In [16]:
print(squad_lat_500[0])

{'instruction': 'Koliko ljudi na jugu Engleske se oslanja na specijaliste NHS Foundation Trust?', 'input': 'Univerzitetska bolnica Sauthempton NHS Fondacija Trast je jedan od najvećih poslodavaca u gradu. Obezbeđuje lokalne bolničke usluge za 500.000 ljudi u području Sauthemptona i specijalizovane regionalne usluge za više od 3 miliona ljudi širom južnog Engleske. Trast poseduje i upravlja opštim bolnicom Sauthempton, bolnicom Princeza Ann i uslugom paliativne nege u kući grofice Mauntbatten, koji je deo lokacije bolnice Moorgrin u selu Vest End, neposredno izvan grada.', 'output': '3 miliona'}


In [17]:
print(squad_cyr_500[0])


{'instruction': 'Који је други термин за пробне теорије које улазе у процес елиминације грешака у науци?', 'input': 'Као одговор на дату проблемску ситуацију (), бројна конкурећа претпоставка, или пробне теорије (), систематски су подвргнуте најрегрузнијим покушајима фалсификације. Овај процес, елиминација грешака (), обавља сличну функцију за науку коју природна селекција обавља за биолошку еволуцију. Теорије које боље преживљавају процес опровергања нису истинитије, већ "прикладније"другим речима, применљивије за ситуацију проблема (). Као последица тога, баш као што биолошка погодност врсте не осигура да ће наставити да живи, тако ни строга тестирање не штити научну теорију од осврнувања у будућности. Ипак, као што се чини да је мотор биолошке еволуције током многих генерација произвео адаптивне особине опремљене да се баве све сложенијим проблемима преживљавања, тако и еволуција теорија кроз научну методу може, по Поперovom мишљењу, одражавати одређену врсту напретка: ка све интере

## 3. Alpaca: Cyrillic and Latin sources

[Cyrillic source](https://huggingface.co/datasets/saillab/alpaca-serbian-cleaned) · [Latin source](https://huggingface.co/datasets/datatab/alpaca-cleaned-serbian-full)

In [18]:
from datasets import load_dataset

# 2. Randomly sample the Alpaca datasets (1,000 examples each)
target_cols = ["instruction", "input", "output"]

ds_saillab = load_dataset("saillab/alpaca-serbian-cleaned", split="train").shuffle(seed=42)
alpaca_cyr_1000 = ds_saillab.select(range(1000))
alpaca_cyr_1000 = alpaca_cyr_1000.remove_columns([c for c in alpaca_cyr_1000.column_names if c not in target_cols])

ds_datatab = load_dataset("datatab/alpaca-cleaned-serbian-full", split="train").shuffle(seed=42)
alpaca_lat_1000 = ds_datatab.select(range(1000))
alpaca_lat_1000 = alpaca_lat_1000.remove_columns([c for c in alpaca_lat_1000.column_names if c not in target_cols])

print(f"Alpaca Ćirilica (saillab): {len(alpaca_cyr_1000)} primera")
print(f"Alpaca Latinica (datatab): {len(alpaca_lat_1000)} primera")

Alpaca Ćirilica (saillab): 1000 primera
Alpaca Latinica (datatab): 1000 primera


In [19]:
print(alpaca_cyr_1000[0])

{'instruction': 'Напишите наслов за новински чланак који говори о напретку у вештачкој интелигенцији.', 'input': 'nan', 'output': ' „Револуционисање будућности: револуционарни напредак у технологији вештачке интелигенције“'}


In [20]:
print(alpaca_lat_1000[0])

{'instruction': 'Preuredite sledeću rečenicu da rečenica bude zanimljivija.', 'input': 'Rano je napustila žurku', 'output': 'Rano je napustila zabavu.'}


## 4. Aya: filter Serbian examples

[Source dataset](https://huggingface.co/datasets/CohereLabs/aya_dataset). Map the source fields to the shared instruction/input/output schema.

In [10]:
from datasets import load_dataset, concatenate_datasets

# 1. Load Aya and filter for Serbian
aya_full = load_dataset("CohereLabs/aya_dataset", split="train")
aya_sr = aya_full.filter(lambda x: x["language_code"] == "srp")

# 2. Map to standard columns (instruction, input, output)
def format_aya(example):
    return {
        "instruction": example["inputs"],
        "input": "",
        "output": example["targets"]
    }

aya_sr = aya_sr.map(format_aya)

# Keep only standard columns
target_cols = ["instruction", "input", "output"]
aya_sr = aya_sr.remove_columns([c for c in aya_sr.column_names if c not in target_cols])

print(f"Učitano primera iz Aya dataset-a: {len(aya_sr)}")



Map: 100%|██████████| 152/152 [00:00<00:00, 2350.67 examples/s]

Učitano primera iz Aya dataset-a: 152


In [11]:
print(aya_sr[0])


{'instruction': 'Шта је ЕФТ, техника Емоционалне слободе? ', 'input': '', 'output': 'ЕФТ (Техника Емоционалне слободе), познатија као тапкање је моћна холистичка исцељујућа техника која решава широк спектар психичких, емотивних и физичких проблема и базира се на принципима традиционалне Кинеске акупресуре и модерне психологије.\n\nЕФТ зову и емоционална акупресура, а основна процедура се изводи тако што тапкамо врховима прстију по одређеним тачкама на нашем телу које представљају крајеве главних енергетских меридијана док смо у исто време ментално фокусирани на проблем који решавамо.'}


## 5. Combine and inspect the training dataset

In [21]:
# Inspect the structure of all source datasets
print(alpaca_cyr_1000, alpaca_lat_1000, squad_cyr_500, squad_lat_500, aya_sr)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 1000
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 1000
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 500
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 500
}) Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 152
})


In [22]:
final_train_dataset = concatenate_datasets([
    squad_lat_500,     # 500 - SQuAD (Latin)
    squad_cyr_500,     # 500 - SQuAD (Cyrillic)
    alpaca_lat_1000,   # 1000 - Alpaca (Latin)
    alpaca_cyr_1000,   # 1000 - Alpaca (Cyrillic)
    aya_sr             # 152 - Aya (Cyrillic)
]).shuffle(seed=42)  # Shuffle all examples


print(f"\nKonačan skup za obuku ima ukupno {len(final_train_dataset)} primera.")
print("\nPrimer iz promešanog skupa:")
print(final_train_dataset[0])


Konačan skup za obuku ima ukupno 3152 primera.

Primer iz promešanog skupa:
{'instruction': 'Koji je najduži suspendirani most u Sjedinjenim Državama?', 'input': 'Most Džordž Vašington je najprometniji most za motorna vozila na svetu, koji povezuje Manhatten sa okrugom Bergen, u Nju Džersiju. Verazano-Narouzski most je najduži suspendirani most u Americi i jedan od najdužih na svetu. Bruklinski most je ikona grada. Kule Bruklinskog mosta sagrađene su od varoviča, granita i Rozendejl cementa, a njihov arhitektonski stil je neo-gotički, sa karakterističnim oštrim lukovima iznad prolaza kroz kamene kule. Ovaj most je takođe bio i najduži suspendioni most na svetu od svog otvaranja do 1903. godine, i prvi je suspendioni most sa čeličnim žicom.', 'output': 'Verazano-Narouzski most'}


## 6. Export the dataset

The output uses JSON Lines format even though its filename ends in `.json`.

In [23]:
# Export the combined dataset, preserving Serbian characters and Cyrillic
file_path = "final_train_dataset.json"

final_train_dataset.to_json(file_path, force_ascii=False)

print(f"Dataset je uspešno sačuvan u: {file_path}")

Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 25.90ba/s]

Dataset je uspešno sačuvan u: final_train_dataset.json


## Reading the recorded results

Saved outputs document the original run. This presentation edit does not rerun the notebook, change its artifacts, or claim that a fresh environment will reproduce the recorded results exactly.